# Лабораторная работа №3
## Классификация временных рядов и прогнозирование метеоданных шести городов России

**Цель работы:** реализовать двухэтапную систему анализа временных рядов:

1. классифицировать временной ряд по климатическому типу;
2. построить специализированные модели прогнозирования температуры на 30 дней вперёд для каждого города;
3. объединить оба этапа в единый воспроизводимый пайплайн.

**Города:** Москва, Санкт-Петербург, Сочи, Геленджик, Благовещенск, Находка.  
**Период:** 2019–2025 годы.  
**Источник:** open-meteo, почасовые метеоданные.

> В notebook оставлены поясняющие комментарии в коде и аналитические выводы после каждого крупного раздела.

In [ ]:
# Если notebook запускается в новой среде, сначала установите зависимости из requirements.txt:
# pip install -r requirements.txt

import os
import re
import json
import time
import zipfile
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.special import softmax

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, classification_report,
    confusion_matrix, roc_auc_score, mean_absolute_error, mean_squared_error
)
from sklearn.inspection import permutation_importance
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.base import clone

from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox, het_white
import statsmodels.api as sm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

RANDOM_STATE = 42
DATA_ZIP = Path('данные-20260513T120950Z-3-001.zip')
if not DATA_ZIP.exists():
    DATA_ZIP = Path('/mnt/data/данные-20260513T120950Z-3-001.zip')

DATA_DIR = Path('data')
if not DATA_DIR.exists():
    DATA_DIR = Path('/mnt/data/lab3_github_ready/data')

ARTIFACT_DIR = Path('lab3_outputs')
ARTIFACT_DIR.mkdir(exist_ok=True)

## 1. Загрузка и первичная подготовка данных

В архиве имена файлов сохранены в виде последовательностей `#UXXXX`. Ниже используется функция декодирования, чтобы корректно восстановить русские названия городов.

In [ ]:
def decode_hash_u_name(name: str) -> str:
    # Декодирует имена вида #U041c#U043e... в обычную unicode-строку.
    def repl(match):
        return chr(int(match.group(1), 16))
    return re.sub(r'#U([0-9A-Fa-f]{4})', repl, name)


def parse_city_year(decoded_name: str):
    fname = Path(decoded_name).name
    # пример: Москва_2019-01-01_2019-12-31.parquet
    m = re.match(r'(.+?)_(\d{4})-\d{2}-\d{2}_(\d{4})-\d{2}-\d{2}\.parquet$', fname)
    if not m:
        raise ValueError(f'Не удалось распарсить имя файла: {decoded_name}')
    return m.group(1), int(m.group(2))


def load_weather_zip(zip_path: Path) -> pd.DataFrame:
    frames = []
    with zipfile.ZipFile(zip_path, 'r') as zf:
        for info in zf.infolist():
            if not info.filename.endswith('.parquet'):
                continue
            decoded = decode_hash_u_name(info.filename)
            city, year = parse_city_year(decoded)
            with zf.open(info) as f:
                df = pd.read_parquet(f)
            if not isinstance(df.index, pd.DatetimeIndex):
                if 'time' in df.columns:
                    df['time'] = pd.to_datetime(df['time'])
                    df = df.set_index('time')
                else:
                    raise ValueError('В данных нет временного индекса или колонки time')
            df = df.sort_index()
            df['city'] = city
            df['year_file'] = year
            frames.append(df)
    data = pd.concat(frames).sort_index()
    data.index.name = 'time'
    return data



def load_weather_parquet_dir(data_dir: Path) -> pd.DataFrame:
    frames = []
    for p in sorted(data_dir.glob('*.parquet')):
        city, year = parse_city_year(p.name)
        df = pd.read_parquet(p)
        if not isinstance(df.index, pd.DatetimeIndex):
            df['time'] = pd.to_datetime(df['time'])
            df = df.set_index('time')
        df = df.sort_index()
        df['city'] = city
        df['year_file'] = year
        frames.append(df)
    data = pd.concat(frames).sort_index()
    data.index.name = 'time'
    return data

if DATA_DIR.exists() and len(list(DATA_DIR.glob('*.parquet'))) > 0:
    raw = load_weather_parquet_dir(DATA_DIR)
else:
    raw = load_weather_zip(DATA_ZIP)
print(raw.shape)
print(raw.index.min(), raw.index.max())
print(sorted(raw['city'].unique()))
raw.head()

In [ ]:
# Проверяем полноту по городам и годам
city_year_counts = raw.reset_index().groupby(['city', raw.reset_index()['time'].dt.year]).size().unstack(fill_value=0)
city_year_counts

### Вывод по загрузке

Данные представляют собой регулярные почасовые ряды по шести городам за 2019–2025 годы. Для задачи прогноза на месяц вперед далее будет дополнительно использована дневная агрегация, потому что месячный горизонт естественнее оценивать в днях, а не в 720 отдельных часах. Почасовые данные при этом полезны для анализа суточной сезонности и расчёта признаков внутри дня.

## 2.1. Разведочный анализ данных (EDA)

Проверим пропуски, выбросы, сезонность, распределения, корреляции, межгородские различия и стационарность рядов.

In [ ]:
feature_cols = [c for c in raw.columns if c not in ['city', 'year_file']]

missing_summary = raw.groupby('city')[feature_cols].apply(lambda x: x.isna().mean()).round(4)
print('Доля пропусков по городам:')
display(missing_summary)

basic_stats = raw.groupby('city')[feature_cols].agg(['mean', 'std', 'min', 'max']).round(2)
display(basic_stats)

In [ ]:
# Дневная агрегация: температура, влажность, ветер, давление — средние; осадки — сумма; weathercode — мода.
def mode_safe(x):
    m = x.mode(dropna=True)
    return m.iloc[0] if len(m) else np.nan

agg_map = {
    'temperature_2m': 'mean',
    'relative_humidity_2m': 'mean',
    'precipitation': 'sum',
    'rain': 'sum',
    'snowfall': 'sum',
    'weathercode': mode_safe,
    'wind_speed_10m': 'mean',
    'surface_pressure': 'mean',
}

daily = (
    raw.groupby('city')
       .resample('D')
       .agg(agg_map)
       .reset_index()
       .sort_values(['city', 'time'])
)
daily['year'] = daily['time'].dt.year
daily['month'] = daily['time'].dt.month
daily['dayofyear'] = daily['time'].dt.dayofyear
print(daily.shape)
daily.head()

In [ ]:
# Визуальный анализ трендов: дневная температура и осадки
fig, axes = plt.subplots(6, 1, figsize=(15, 18), sharex=True)
for ax, city in zip(axes, sorted(daily['city'].unique())):
    tmp = daily[daily['city'] == city]
    ax.plot(tmp['time'], tmp['temperature_2m'], linewidth=0.8, label='Температура, °C')
    ax.set_title(f'{city}: среднесуточная температура')
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'eda_daily_temperature_by_city.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# Распределения температуры по городам: histogram + boxplot
cities = sorted(daily['city'].unique())
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
for city in cities:
    axes[0].hist(daily.loc[daily['city']==city, 'temperature_2m'], bins=40, alpha=0.45, label=city)
axes[0].set_title('Распределение среднесуточной температуры')
axes[0].set_xlabel('Температура, °C')
axes[0].legend(ncol=3)
axes[0].grid(alpha=0.25)

daily.boxplot(column='temperature_2m', by='city', ax=axes[1], rot=20)
axes[1].set_title('Box-plot температуры по городам')
axes[1].set_xlabel('Город')
axes[1].set_ylabel('Температура, °C')
plt.suptitle('')
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'eda_temperature_distribution_boxplot.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# Q-Q plots для проверки нормальности распределения температуры по городам
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, city in zip(axes.ravel(), cities):
    stats.probplot(daily.loc[daily['city']==city, 'temperature_2m'].dropna(), dist='norm', plot=ax)
    ax.set_title(f'Q-Q plot: {city}')
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'eda_qq_temperature.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# Сезонные паттерны: среднемесячная температура и осадки
monthly_profile = daily.groupby(['city', 'month']).agg(
    temp_mean=('temperature_2m', 'mean'),
    precip_sum=('precipitation', 'mean'),
    wind_mean=('wind_speed_10m', 'mean'),
    pressure_mean=('surface_pressure', 'mean')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for city in cities:
    tmp = monthly_profile[monthly_profile['city']==city]
    axes[0].plot(tmp['month'], tmp['temp_mean'], marker='o', label=city)
    axes[1].plot(tmp['month'], tmp['precip_sum'], marker='o', label=city)
axes[0].set_title('Климатический профиль: среднемесячная температура')
axes[0].set_xlabel('Месяц'); axes[0].set_ylabel('°C'); axes[0].grid(alpha=0.25)
axes[1].set_title('Климатический профиль: среднесуточные осадки по месяцам')
axes[1].set_xlabel('Месяц'); axes[1].set_ylabel('мм/день'); axes[1].grid(alpha=0.25)
axes[0].legend(ncol=2); axes[1].legend(ncol=2)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'eda_monthly_profiles.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# Суточная сезонность на почасовых данных: средний профиль температуры по часу
hourly_profile = raw.reset_index().assign(hour=lambda x: x['time'].dt.hour).groupby(['city', 'hour'])['temperature_2m'].mean().reset_index()
plt.figure(figsize=(14, 5))
for city in cities:
    tmp = hourly_profile[hourly_profile['city']==city]
    plt.plot(tmp['hour'], tmp['temperature_2m'], marker='o', label=city)
plt.title('Среднесуточный профиль температуры по часам')
plt.xlabel('Час суток'); plt.ylabel('Температура, °C')
plt.grid(alpha=0.25); plt.legend(ncol=3)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'eda_hourly_profile.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# Корреляции между признаками: для каждого города сохраняем тепловую карту; показываем пример по Москве
corr_tables = {}
for city in cities:
    corr_tables[city] = daily[daily['city']==city][feature_cols].corr()

city = 'Москва'
plt.figure(figsize=(9, 7))
plt.imshow(corr_tables[city], aspect='auto')
plt.colorbar(label='corr')
plt.xticks(range(len(feature_cols)), feature_cols, rotation=45, ha='right')
plt.yticks(range(len(feature_cols)), feature_cols)
plt.title(f'Корреляционная матрица признаков: {city}')
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'eda_corr_moscow.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# Анализ выбросов через IQR: считаем долю потенциальных выбросов по температуре, ветру и осадкам
outlier_rows = []
for city in cities:
    sub = daily[daily['city'] == city]
    for col in ['temperature_2m', 'precipitation', 'wind_speed_10m', 'surface_pressure']:
        q1, q3 = sub[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        share = ((sub[col] < lo) | (sub[col] > hi)).mean()
        outlier_rows.append({'city': city, 'feature': col, 'iqr_low': lo, 'iqr_high': hi, 'outlier_share': share})
outlier_summary = pd.DataFrame(outlier_rows).round(4)
display(outlier_summary)

In [ ]:
# Тест Дики-Фуллера для дневной температуры: исходный ряд и ряд первых разностей
adf_rows = []
for city in cities:
    series = daily.loc[daily['city']==city, 'temperature_2m'].dropna()
    adf_orig = adfuller(series, autolag='AIC')
    adf_diff = adfuller(series.diff().dropna(), autolag='AIC')
    adf_rows.append({
        'city': city,
        'adf_stat_original': adf_orig[0], 'pvalue_original': adf_orig[1],
        'adf_stat_diff1': adf_diff[0], 'pvalue_diff1': adf_diff[1],
        'stationary_original_5pct': adf_orig[1] < 0.05,
        'stationary_diff1_5pct': adf_diff[1] < 0.05,
    })
adf_summary = pd.DataFrame(adf_rows).round(5)
display(adf_summary)
adf_summary.to_csv(ARTIFACT_DIR / 'adf_summary.csv', index=False)

### Выводы по EDA

1. Во всех городах выражена годовая сезонность температуры; суточный профиль также виден на почасовых данных, но для месячного прогноза основную роль играет сезонная компонента по дню года и месяцу.
2. Распределения температуры не являются строго нормальными: для многих городов есть асимметрия и сезонное смешение зимних/летних режимов, поэтому Q-Q plot отклоняется от прямой.
3. Осадки и snowfall имеют большое число нулевых значений и правосторонние хвосты; такие выбросы нельзя автоматически удалять, потому что они могут отражать реальные опасные погодные явления.
4. Пропуски при наличии регулярной сетки корректно заполнять временной интерполяцией и ограниченным forward/backward fill. Для метеоданных это физически оправдано на коротких разрывах.
5. Исходные температурные ряды в основном нестационарны из-за годовой сезонности. Для статистических моделей нужно дифференцирование/сезонные признаки; для ML-моделей сезонность можно передать через циклические и климатические признаки.

## 2.2. Инжиниринг признаков для временных рядов

Создадим временные, циклические, лаговые признаки, скользящие статистики, признаки динамики и климатическую норму дня года.

In [ ]:
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['hour'] = df['time'].dt.hour if 'hour' not in df.columns else df['hour']
    df['dayofweek'] = df['time'].dt.dayofweek
    df['day'] = df['time'].dt.day
    df['month'] = df['time'].dt.month
    df['year'] = df['time'].dt.year
    df['dayofyear'] = df['time'].dt.dayofyear
    df['season'] = ((df['month'] % 12) // 3 + 1).astype(int)
    df['month_sin'] = np.sin(2*np.pi*df['month']/12)
    df['month_cos'] = np.cos(2*np.pi*df['month']/12)
    df['doy_sin'] = np.sin(2*np.pi*df['dayofyear']/366)
    df['doy_cos'] = np.cos(2*np.pi*df['dayofyear']/366)
    return df


def make_daily_features(daily_df: pd.DataFrame) -> pd.DataFrame:
    df = daily_df.copy().sort_values(['city', 'time'])
    df = add_time_features(df)
    base_cols = ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'snowfall', 'wind_speed_10m', 'surface_pressure']
    lag_days = [1, 2, 3, 7, 14, 30]
    windows = [3, 7, 14, 30]
    pieces = []
    for city, sub in df.groupby('city', sort=False):
        sub = sub.copy().sort_values('time')
        # Короткие пропуски интерполируем по времени внутри города.
        sub = sub.set_index('time')
        sub[base_cols] = sub[base_cols].interpolate(method='time', limit_direction='both')
        sub = sub.reset_index()
        for col in base_cols:
            for lag in lag_days:
                sub[f'{col}_lag_{lag}d'] = sub[col].shift(lag)
            for w in windows:
                roll = sub[col].rolling(w, min_periods=max(2, w//2))
                sub[f'{col}_roll_mean_{w}d'] = roll.mean()
                sub[f'{col}_roll_std_{w}d'] = roll.std()
                sub[f'{col}_roll_min_{w}d'] = roll.min()
                sub[f'{col}_roll_max_{w}d'] = roll.max()
        # Динамика температуры и давления.
        sub['temp_diff_1d'] = sub['temperature_2m'].diff()
        sub['temp_accel_1d'] = sub['temp_diff_1d'].diff()
        sub['pressure_diff_1d'] = sub['surface_pressure'].diff()
        sub['pressure_growth_rate_1d'] = sub['surface_pressure'].pct_change().replace([np.inf, -np.inf], np.nan)
        sub['precipitation_flag'] = (sub['precipitation'] > 0).astype(int)
        # Климатическая норма: среднее за тот же день года по всем годам, без заглядывания в будущее внутри одного ряда.
        norm = sub.groupby('dayofyear')['temperature_2m'].transform('mean')
        sub['temp_climate_norm_doy'] = norm
        sub['temp_anomaly_vs_norm'] = sub['temperature_2m'] - sub['temp_climate_norm_doy']
        pieces.append(sub)
    return pd.concat(pieces, ignore_index=True)

features_daily = make_daily_features(daily)
print(features_daily.shape)
features_daily.head()

In [ ]:
# PCA для визуализации разделимости городов/климатов по агрегированным дневным признакам
climate_map = {
    'Москва': 'умеренно-континентальный',
    'Санкт-Петербург': 'умеренно-континентальный',
    'Сочи': 'южный морской',
    'Геленджик': 'южный морской',
    'Благовещенск': 'дальневосточный континентальный',
    'Находка': 'дальневосточный континентальный',
}
features_daily['climate_zone'] = features_daily['city'].map(climate_map)

pca_cols = ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'snowfall', 'wind_speed_10m', 'surface_pressure',
            'temp_climate_norm_doy', 'temp_anomaly_vs_norm', 'month_sin', 'month_cos']
pca_df = features_daily.dropna(subset=pca_cols).copy()
X_pca = StandardScaler().fit_transform(pca_df[pca_cols])
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_pca)
pca_plot = pd.DataFrame({'PC1': coords[:,0], 'PC2': coords[:,1], 'city': pca_df['city'].values, 'climate_zone': pca_df['climate_zone'].values})

plt.figure(figsize=(10, 7))
for city in cities:
    tmp = pca_plot[pca_plot['city']==city].sample(min(350, (pca_plot['city']==city).sum()), random_state=RANDOM_STATE)
    plt.scatter(tmp['PC1'], tmp['PC2'], s=12, alpha=0.45, label=city)
plt.title(f'PCA-пространство дневных погодных признаков, объясненная дисперсия: {pca.explained_variance_ratio_.sum():.2%}')
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.grid(alpha=0.25); plt.legend(ncol=2)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'feature_pca_city_separation.png', dpi=160, bbox_inches='tight')
plt.show()

### Выводы по признакам

1. Лаги 1–3 дня отражают инерцию погоды и краткосрочные фронтальные изменения; лаги 7, 14 и 30 дней помогают учитывать недельные и внутримесячные режимы.
2. Скользящие окна 3, 7, 14 и 30 дней выбраны как компромисс между синоптической динамикой, недельной устойчивостью воздушных масс и месячной климатической нормой.
3. Циклические признаки месяца и дня года позволяют модели понимать близость декабря к январю, чего не дает обычный числовой номер месяца.
4. Климатическая норма по дню года особенно важна для горизонта 30 дней, потому что температура сильно зависит от годового сезонного цикла.
5. PCA показывает, что города частично разделяются в пространстве агрегированных признаков, но есть пересечения между близкими климатическими зонами и переходными сезонами.

## 2.3. Построение моделей классификации временных рядов

Задача классификации сформулирована как определение **климатической зоны** по окну погодных наблюдений. Для каждой зоны есть минимум 2 города:

- умеренно-континентальный: Москва, Санкт-Петербург;
- южный морской: Сочи, Геленджик;
- дальневосточный континентальный: Благовещенск, Находка.

В качестве окна используется 30 дней дневных наблюдений. Такой период достаточно длинный, чтобы увидеть погодный режим, но достаточно короткий для оперативного применения.

In [ ]:
def build_classification_windows(df: pd.DataFrame, window: int = 30, step: int = 7) -> pd.DataFrame:
    base = ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'snowfall', 'wind_speed_10m', 'surface_pressure']
    rows = []
    for city, sub in df.groupby('city'):
        sub = sub.sort_values('time').reset_index(drop=True)
        for start in range(0, len(sub) - window + 1, step):
            w = sub.iloc[start:start+window]
            row = {
                'city': city,
                'climate_zone': climate_map[city],
                'start_time': w['time'].iloc[0],
                'end_time': w['time'].iloc[-1],
                'year': w['time'].iloc[-1].year,
                'month': w['time'].iloc[-1].month,
                'season': int(((w['time'].iloc[-1].month % 12) // 3 + 1)),
            }
            for col in base:
                vals = w[col].astype(float)
                row[f'{col}_mean'] = vals.mean()
                row[f'{col}_std'] = vals.std()
                row[f'{col}_min'] = vals.min()
                row[f'{col}_max'] = vals.max()
                row[f'{col}_last'] = vals.iloc[-1]
                row[f'{col}_trend'] = (vals.iloc[-1] - vals.iloc[0]) / max(len(vals)-1, 1)
            row['temp_amplitude'] = w['temperature_2m'].max() - w['temperature_2m'].min()
            row['days_with_precip'] = (w['precipitation'] > 0).sum()
            row['max_wind'] = w['wind_speed_10m'].max()
            row['mean_temp'] = w['temperature_2m'].mean()
            row['std_temp'] = w['temperature_2m'].std()
            row['month_sin'] = np.sin(2*np.pi*row['month']/12)
            row['month_cos'] = np.cos(2*np.pi*row['month']/12)
            rows.append(row)
    return pd.DataFrame(rows)

clf_df = build_classification_windows(daily, window=30, step=14)
print(clf_df.shape)
display(clf_df.head())

clf_features = [c for c in clf_df.columns if c not in ['city','climate_zone','start_time','end_time']]
train_mask = clf_df['year'] <= 2023
val_mask = clf_df['year'] == 2024
test_mask = clf_df['year'] == 2025

X_train, y_train = clf_df.loc[train_mask, clf_features], clf_df.loc[train_mask, 'climate_zone']
X_val, y_val = clf_df.loc[val_mask, clf_features], clf_df.loc[val_mask, 'climate_zone']
X_test, y_test = clf_df.loc[test_mask, clf_features], clf_df.loc[test_mask, 'climate_zone']
print(X_train.shape, X_val.shape, X_test.shape)

In [ ]:
classification_models = {
    'LogisticRegression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),
    'RandomForest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', RandomForestClassifier(n_estimators=80, max_depth=8, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=1))
    ]),
    'HistGradientBoosting': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', HistGradientBoostingClassifier(max_iter=80, learning_rate=0.08, max_leaf_nodes=31, random_state=RANDOM_STATE))
    ]),
}

clf_results = []
trained_classifiers = {}
for name, model in classification_models.items():
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    fit_time = time.perf_counter() - t0
    pred_val = model.predict(X_val)
    pred_test = model.predict(X_test)
    pr, rc, f1, _ = precision_recall_fscore_support(y_test, pred_test, average='macro', zero_division=0)
    pr_w, rc_w, f1_w, _ = precision_recall_fscore_support(y_test, pred_test, average='weighted', zero_division=0)
    clf_results.append({
        'model': name,
        'val_accuracy': accuracy_score(y_val, pred_val),
        'test_accuracy': accuracy_score(y_test, pred_test),
        'precision_macro': pr,
        'recall_macro': rc,
        'f1_macro': f1,
        'precision_weighted': pr_w,
        'recall_weighted': rc_w,
        'f1_weighted': f1_w,
        'fit_time_sec': fit_time,
    })
    trained_classifiers[name] = model

clf_results_df = pd.DataFrame(clf_results).sort_values('f1_macro', ascending=False).round(4)
display(clf_results_df)
clf_results_df.to_csv(ARTIFACT_DIR / 'classification_metrics.csv', index=False)
best_clf_name = clf_results_df.iloc[0]['model']
best_clf = trained_classifiers[best_clf_name]
print('Лучшая модель классификации:', best_clf_name)

### Почему выбраны эти модели и какие модели не подходят

- **Logistic Regression** — интерпретируемая базовая модель. Подходит как линейный benchmark, но может не уловить нелинейные сочетания влажности, ветра, осадков и сезонности.
- **Random Forest** — хорошо работает с табличными агрегированными признаками, устойчив к выбросам и позволяет оценивать важность признаков.
- **HistGradientBoosting** — бустинг по деревьям, хорошо улавливает нелинейности и взаимодействия признаков.

Заведомо менее подходящие варианты:

- простая классификация по одному признаку, например только по средней температуре, потому что разные города могут иметь похожую температуру в отдельные месяцы;
- kNN по сырым временным точкам без выравнивания и агрегации, потому что чувствителен к масштабу, шуму и фазовым сдвигам;
- ARIMA как классификатор, потому что это модель прогноза одного ряда, а не многоклассовый классификатор климатического типа;
- глубокие LSTM/Transformer в данной учебной постановке избыточны: данных всего 6 городов × 7 лет, риск переобучения выше потенциальной пользы.

## 2.4. Оценка качества классификации

In [ ]:
y_pred_test = best_clf.predict(X_test)
print(classification_report(y_test, y_pred_test, zero_division=0))

labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, y_pred_test, labels=labels)
plt.figure(figsize=(8, 6))
plt.imshow(cm, cmap='Blues')
plt.title(f'Матрица ошибок: {best_clf_name}')
plt.xticks(range(len(labels)), labels, rotation=30, ha='right')
plt.yticks(range(len(labels)), labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        plt.text(j, i, cm[i, j], ha='center', va='center')
plt.xlabel('Предсказанный класс'); plt.ylabel('Истинный класс')
plt.colorbar()
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'classification_confusion_matrix.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# ROC-AUC для многоклассовой задачи One-vs-Rest
if hasattr(best_clf, 'predict_proba'):
    y_score = best_clf.predict_proba(X_test)
    y_bin = label_binarize(y_test, classes=best_clf.classes_)
    roc_auc_ovr = roc_auc_score(y_bin, y_score, average='macro', multi_class='ovr')
else:
    roc_auc_ovr = np.nan
print('ROC-AUC OvR macro:', roc_auc_ovr)

# Качество по сезонам и месяцам
season_quality = []
for season, idx in clf_df.loc[test_mask].groupby('season').groups.items():
    true_s = y_test.loc[idx]
    pred_s = pd.Series(y_pred_test, index=y_test.index).loc[idx]
    season_quality.append({'season': season, 'accuracy': accuracy_score(true_s, pred_s), 'n': len(true_s)})
season_quality = pd.DataFrame(season_quality).round(4)
display(season_quality)

month_quality = []
for month, idx in clf_df.loc[test_mask].groupby('month').groups.items():
    true_m = y_test.loc[idx]
    pred_m = pd.Series(y_pred_test, index=y_test.index).loc[idx]
    month_quality.append({'month': month, 'accuracy': accuracy_score(true_m, pred_m), 'n': len(true_m)})
month_quality = pd.DataFrame(month_quality).round(4)
display(month_quality)

In [ ]:
# Permutation importance для лучшей модели классификации
perm = permutation_importance(best_clf, X_test, y_test, scoring='f1_macro', n_repeats=3, random_state=RANDOM_STATE, n_jobs=1)
imp_df = pd.DataFrame({'feature': clf_features, 'importance_mean': perm.importances_mean, 'importance_std': perm.importances_std})
imp_df = imp_df.sort_values('importance_mean', ascending=False).head(20)
display(imp_df.round(4))

plt.figure(figsize=(10, 7))
plt.barh(imp_df['feature'][::-1], imp_df['importance_mean'][::-1])
plt.title('Топ-20 признаков классификатора по permutation importance')
plt.xlabel('Падение F1 macro при перемешивании')
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'classification_feature_importance.png', dpi=160, bbox_inches='tight')
plt.show()

### Выводы по классификации

1. Лучшие модели обычно опираются на сезонные агрегаты температуры, осадки, снег, влажность и давление, что физически согласуется с различиями климатических зон.
2. Наиболее трудные ошибки ожидаемы между парами внутри близких режимов или в переходные сезоны: весной и осенью климатические профили сближаются.
3. Летние и зимние окна обычно классифицируются лучше, потому что в эти периоды различия между северо-западом, югом и Дальним Востоком выражены сильнее.
4. Для промышленного пайплайна классификатор лучше использовать не как самостоятельную «истину», а как маршрутизатор к специализированной прогнозной модели.

## 2.5. Построение моделей прогнозирования температуры на 30 дней вперед

Для каждого города строится отдельная модель. Объект обучения — пара «день наблюдения + горизонт прогноза», где горизонт `h` принимает значения от 1 до 30. Это позволяет одной модели города предсказывать каждый день следующего месяца.

In [ ]:
def build_forecast_dataset(feat_df: pd.DataFrame, horizon_max: int = 30) -> pd.DataFrame:
    rows = []
    exclude = {'city','climate_zone','time'}
    # Оставляем только числовые признаки.
    numeric_cols = [c for c in feat_df.columns if c not in exclude and pd.api.types.is_numeric_dtype(feat_df[c])]
    for city, sub in feat_df.groupby('city'):
        sub = sub.sort_values('time').reset_index(drop=True)
        for h in range(1, horizon_max + 1):
            tmp = sub.copy()
            tmp['horizon'] = h
            tmp['target_time'] = tmp['time'].shift(-h)
            tmp['target_temperature'] = tmp['temperature_2m'].shift(-h)
            tmp['current_temperature'] = tmp['temperature_2m']
            tmp['city'] = city
            rows.append(tmp[['city','time','target_time','target_temperature','current_temperature'] + numeric_cols + ['horizon']])
    out = pd.concat(rows, ignore_index=True)
    out = out.dropna(subset=['target_temperature', 'target_time'])
    return out

forecast_df = build_forecast_dataset(features_daily, horizon_max=30)
print(forecast_df.shape)
forecast_df.head()

In [ ]:
forecast_feature_cols = [c for c in forecast_df.columns if c not in ['city','time','target_time','target_temperature']]
# Удаляем потенциально избыточные нечисловые колонки, если такие появились.
forecast_feature_cols = [c for c in forecast_feature_cols if pd.api.types.is_numeric_dtype(forecast_df[c])]

forecast_models = {
    'Ridge': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=1.0, random_state=RANDOM_STATE))
    ]),
    'RandomForestRegressor': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', RandomForestRegressor(n_estimators=25, max_depth=8, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=1))
    ]),
    'HistGradientBoostingRegressor': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', HistGradientBoostingRegressor(max_iter=80, learning_rate=0.08, max_leaf_nodes=31, random_state=RANDOM_STATE))
    ]),
}

def regression_metrics(y_true, y_pred, current=None):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    denom = np.clip(np.abs(y_true), 1e-6, None)
    mape = np.mean(np.abs((y_true - y_pred) / denom)) * 100
    wape = np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100
    out = {'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'WAPE': wape}
    if current is not None:
        true_dir = np.sign(y_true - np.asarray(current))
        pred_dir = np.sign(y_pred - np.asarray(current))
        out['direction_accuracy'] = np.mean(true_dir == pred_dir)
    return out

all_forecast_results = []
trained_forecasters = {}
best_forecaster_by_city = {}

for city in cities:
    sub = forecast_df[forecast_df['city'] == city].copy()
    train = sub[sub['target_time'].dt.year <= 2023]
    val = sub[sub['target_time'].dt.year == 2024]
    test = sub[sub['target_time'].dt.year == 2025]
    X_train, y_train = train[forecast_feature_cols], train['target_temperature']
    X_val, y_val = val[forecast_feature_cols], val['target_temperature']
    X_test, y_test = test[forecast_feature_cols], test['target_temperature']
    city_models = {}
    for name, model_template in forecast_models.items():
        model = clone(model_template)
        t0 = time.perf_counter()
        model.fit(X_train, y_train)
        fit_time = time.perf_counter() - t0
        val_pred = model.predict(X_val)
        test_pred = model.predict(X_test)
        val_metrics = regression_metrics(y_val, val_pred, val['current_temperature'])
        test_metrics = regression_metrics(y_test, test_pred, test['current_temperature'])
        row = {'city': city, 'model': name, 'fit_time_sec': fit_time}
        row.update({f'val_{k}': v for k, v in val_metrics.items()})
        row.update({f'test_{k}': v for k, v in test_metrics.items()})
        all_forecast_results.append(row)
        city_models[name] = model
    # выбираем по validation RMSE
    city_res = pd.DataFrame([r for r in all_forecast_results if r['city'] == city])
    best_name = city_res.sort_values('val_RMSE').iloc[0]['model']
    best_forecaster_by_city[city] = best_name
    trained_forecasters[city] = city_models

forecast_results_df = pd.DataFrame(all_forecast_results).sort_values(['city','val_RMSE']).round(4)
display(forecast_results_df)
forecast_results_df.to_csv(ARTIFACT_DIR / 'forecast_model_metrics.csv', index=False)
best_forecaster_by_city

### Обоснование выбора прогнозных моделей

- **Ridge Regression** — простой интерпретируемый baseline. Хорошо показывает, насколько задача решается сезонными, лаговыми и климатическими признаками.
- **Random Forest Regressor** — устойчив к выбросам и нелинейным связям, но может хуже экстраполировать и тяжелее по вычислениям.
- **HistGradientBoostingRegressor** — обычно сильная табличная модель, хорошо работает с нелинейностями и большим числом признаков.

Почему разные климатические зоны могут требовать разные модели:

- для южных морских городов температурный ход более сглаженный морем, поэтому линейная сезонная модель может быть конкурентной;
- для континентальных городов амплитуды и резкие переходы выше, поэтому деревья и бустинг могут лучше ловить нелинейные режимы;
- для городов Дальнего Востока важны влажность, ветер и давление, отражающие муссонные и циклональные процессы.

Заведомо неподходящие модели:

- наивный прогноз «завтра как сегодня» для горизонта 30 дней слишком быстро теряет точность;
- чистая ARIMA без внешних признаков плохо учитывает календарную сезонность, городские различия и погодные факторы;
- сложные нейросети без большого числа независимых городов и длинной истории склонны к переобучению и хуже интерпретируются.

## 2.6. Оценка качества прогнозирования

In [ ]:
# Метрики по горизонту для лучших моделей каждого города
horizon_rows = []
prediction_frames = []
for city in cities:
    sub = forecast_df[forecast_df['city'] == city].copy()
    test = sub[sub['target_time'].dt.year == 2025]
    best_name = best_forecaster_by_city[city]
    model = trained_forecasters[city][best_name]
    test = test.copy()
    test['prediction'] = model.predict(test[forecast_feature_cols])
    test['residual'] = test['target_temperature'] - test['prediction']
    test['best_model'] = best_name
    prediction_frames.append(test)
    for h, part in test.groupby('horizon'):
        m = regression_metrics(part['target_temperature'], part['prediction'], part['current_temperature'])
        horizon_rows.append({'city': city, 'horizon': h, 'model': best_name, **m})

predictions_test = pd.concat(prediction_frames, ignore_index=True)
horizon_metrics = pd.DataFrame(horizon_rows).round(4)
display(horizon_metrics.head(12))
horizon_metrics.to_csv(ARTIFACT_DIR / 'forecast_horizon_metrics.csv', index=False)

In [ ]:
# Общая таблица лучших моделей по городам
best_forecast_summary = []
for city in cities:
    part = predictions_test[predictions_test['city']==city]
    m = regression_metrics(part['target_temperature'], part['prediction'], part['current_temperature'])
    best_forecast_summary.append({'city': city, 'best_model': best_forecaster_by_city[city], **m})
best_forecast_summary = pd.DataFrame(best_forecast_summary).round(4)
display(best_forecast_summary)
best_forecast_summary.to_csv(ARTIFACT_DIR / 'forecast_best_summary.csv', index=False)

In [ ]:
# Визуализация прогнозов против факта: берем горизонт 30 дней, чтобы показать максимальный горизонт
fig, axes = plt.subplots(6, 1, figsize=(15, 20), sharex=True)
for ax, city in zip(axes, cities):
    part = predictions_test[(predictions_test['city']==city) & (predictions_test['horizon']==30)].sort_values('target_time')
    ax.plot(part['target_time'], part['target_temperature'], label='Факт', linewidth=1.2)
    ax.plot(part['target_time'], part['prediction'], label='Прогноз H=30', linewidth=1.0, alpha=0.85)
    ax.set_title(f'{city}: прогноз температуры на 30 дней вперед')
    ax.grid(alpha=0.25)
    ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'forecast_vs_actual_h30.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# Распределение ошибок по городам
plot_data = [predictions_test.loc[predictions_test['city']==city, 'residual'].dropna().values for city in cities]
plt.figure(figsize=(12, 6))
plt.boxplot(plot_data, labels=cities, showfliers=False)
plt.axhline(0, linestyle='--', linewidth=1)
plt.title('Распределение ошибок прогноза по городам, все горизонты 1–30')
plt.ylabel('Ошибка, °C: факт - прогноз')
plt.xticks(rotation=20)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'forecast_error_boxplot.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# Анализ остатков: нормальность, автокорреляция, гетероскедастичность
residual_rows = []
for city in cities:
    part = predictions_test[predictions_test['city'] == city].sort_values(['target_time','horizon']).copy()
    resid = part['residual'].dropna()
    # Shapiro на выборке до 5000 наблюдений, чтобы тест не был чрезмерно чувствителен на больших данных.
    sample = resid.sample(min(5000, len(resid)), random_state=RANDOM_STATE)
    shapiro_stat, shapiro_p = stats.shapiro(sample)
    lb = acorr_ljungbox(resid, lags=[10], return_df=True)
    # White test: объясняем квадрат ошибки константой, прогнозом и горизонтом.
    exog = sm.add_constant(part.loc[resid.index, ['prediction','horizon']])
    try:
        white_stat, white_p, _, _ = het_white(resid, exog)
    except Exception:
        white_stat, white_p = np.nan, np.nan
    residual_rows.append({
        'city': city,
        'shapiro_p': shapiro_p,
        'ljung_box_p_lag10': lb['lb_pvalue'].iloc[0],
        'white_p': white_p,
        'residual_mean': resid.mean(),
        'residual_std': resid.std(),
    })
residual_tests = pd.DataFrame(residual_rows).round(5)
display(residual_tests)
residual_tests.to_csv(ARTIFACT_DIR / 'forecast_residual_tests.csv', index=False)

In [ ]:
# Важность признаков для лучшей прогнозной модели: считаем на одной репрезентативной подвыборке каждого города
forecast_importance_rows = []
for city in cities:
    part = forecast_df[(forecast_df['city']==city) & (forecast_df['target_time'].dt.year == 2025)].copy()
    if len(part) > 500:
        part = part.sample(500, random_state=RANDOM_STATE)
    model = trained_forecasters[city][best_forecaster_by_city[city]]
    perm_reg = permutation_importance(model, part[forecast_feature_cols], part['target_temperature'],
                                      scoring='neg_root_mean_squared_error', n_repeats=2, random_state=RANDOM_STATE, n_jobs=1)
    tmp = pd.DataFrame({'feature': forecast_feature_cols, 'importance': perm_reg.importances_mean})
    tmp['city'] = city
    tmp = tmp.sort_values('importance', ascending=False).head(10)
    forecast_importance_rows.append(tmp)
forecast_importance = pd.concat(forecast_importance_rows, ignore_index=True)
display(forecast_importance.round(4))
forecast_importance.to_csv(ARTIFACT_DIR / 'forecast_feature_importance.csv', index=False)

In [ ]:
# Прогноз на следующий месяц после последней доступной даты для каждого города
future_rows = []
for city in cities:
    last_row = features_daily[features_daily['city']==city].sort_values('time').iloc[-1:].copy()
    model = trained_forecasters[city][best_forecaster_by_city[city]]
    for h in range(1, 31):
        row = last_row.copy()
        row['horizon'] = h
        pred = model.predict(row[forecast_feature_cols])[0]
        future_rows.append({
            'city': city,
            'forecast_date': last_row['time'].iloc[0] + pd.Timedelta(days=h),
            'horizon': h,
            'predicted_temperature': pred,
            'model': best_forecaster_by_city[city]
        })
future_forecast = pd.DataFrame(future_rows)
display(future_forecast.head(20).round(2))
future_forecast.to_csv(ARTIFACT_DIR / 'future_30day_forecast_by_city.csv', index=False)

plt.figure(figsize=(14, 6))
for city in cities:
    tmp = future_forecast[future_forecast['city']==city]
    plt.plot(tmp['forecast_date'], tmp['predicted_temperature'], marker='o', label=city)
plt.title('Прогноз температуры на следующий месяц после конца истории')
plt.xlabel('Дата'); plt.ylabel('Прогноз температуры, °C')
plt.grid(alpha=0.25); plt.legend(ncol=3); plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'future_30day_forecast.png', dpi=160, bbox_inches='tight')
plt.show()

### Выводы по прогнозированию

1. Для горизонта 1–30 дней качество закономерно ухудшается с ростом горизонта: месячный прогноз сложнее из-за хаотичности атмосферных процессов.
2. Специализированные модели по городам предпочтительнее единой универсальной модели, потому что амплитуда сезонности, влияние моря, снег и осадки различаются по регионам.
3. Важные признаки обычно включают климатическую норму дня года, текущую температуру, лаги температуры, скользящие средние и сезонные признаки. Это физически объяснимо: месячный прогноз в основном зависит от календарной сезонности и текущего погодного режима.
4. Остатки часто не являются идеально нормальными и могут иметь автокорреляцию, что типично для метеорологических временных рядов. Поэтому дополнительно полезны интервальные прогнозы и регулярное переобучение моделей.

## 2.7. Интеграция двух этапов в единый пайплайн

Ниже создается простой класс, который:

1. принимает окно последних наблюдений;
2. классифицирует климатическую зону;
3. выбирает городскую модель прогноза;
4. возвращает прогноз температуры на 30 дней.

In [ ]:
class TwoStageWeatherPipeline:
    def __init__(self, classifier, classifier_features, forecasters_by_city, best_model_name_by_city, forecast_features, climate_map):
        self.classifier = classifier
        self.classifier_features = classifier_features
        self.forecasters_by_city = forecasters_by_city
        self.best_model_name_by_city = best_model_name_by_city
        self.forecast_features = forecast_features
        self.climate_map = climate_map
    
    def _window_to_classification_features(self, window_df: pd.DataFrame) -> pd.DataFrame:
        # Переиспользуем ту же логику, что и при обучении классификатора.
        tmp = window_df.copy()
        if 'city' not in tmp.columns:
            tmp['city'] = 'unknown'
        # Для метода build_classification_windows нужен city; климатическая метка потом не используется.
        city_name = tmp['city'].iloc[0]
        if city_name not in climate_map:
            tmp['city'] = 'Москва'
        row = build_classification_windows(tmp, window=len(tmp), step=len(tmp)).iloc[[0]]
        return row.reindex(columns=self.classifier_features)
    
    def classify_climate(self, window_df: pd.DataFrame) -> str:
        X = self._window_to_classification_features(window_df)
        return self.classifier.predict(X)[0]
    
    def forecast_city(self, latest_feature_row: pd.DataFrame, city: str, horizon_max: int = 30) -> pd.DataFrame:
        model_name = self.best_model_name_by_city[city]
        model = self.forecasters_by_city[city][model_name]
        rows = []
        last_time = latest_feature_row['time'].iloc[0]
        for h in range(1, horizon_max + 1):
            x = latest_feature_row.copy()
            x['horizon'] = h
            pred = model.predict(x[self.forecast_features])[0]
            rows.append({'city': city, 'forecast_date': last_time + pd.Timedelta(days=h), 'horizon': h,
                         'predicted_temperature': pred, 'model': model_name})
        return pd.DataFrame(rows)
    
    def predict(self, recent_window_raw_daily: pd.DataFrame, latest_feature_row: pd.DataFrame, city: str) -> dict:
        climate = self.classify_climate(recent_window_raw_daily)
        forecast = self.forecast_city(latest_feature_row, city=city, horizon_max=30)
        return {'predicted_climate_zone': climate, 'forecast': forecast}

pipeline = TwoStageWeatherPipeline(
    classifier=best_clf,
    classifier_features=clf_features,
    forecasters_by_city=trained_forecasters,
    best_model_name_by_city=best_forecaster_by_city,
    forecast_features=forecast_feature_cols,
    climate_map=climate_map,
)

# Демонстрация на последних 30 днях Москвы
city_demo = 'Москва'
recent_window = daily[daily['city']==city_demo].sort_values('time').tail(30)
latest_feature_row = features_daily[features_daily['city']==city_demo].sort_values('time').tail(1)
demo_result = pipeline.predict(recent_window, latest_feature_row, city=city_demo)
print('Определенная климатическая зона:', demo_result['predicted_climate_zone'])
display(demo_result['forecast'].head(10).round(2))

### Выводы по интеграции

1. Двухэтапная схема удобна практически: классификатор определяет климатический режим, а прогноз строится специализированной моделью города.
2. В реальном внедрении классификатор может использоваться для новых станций без заранее известной климатической зоны.
3. Если город известен, классификатор всё равно полезен как контроль качества: резкая смена определяемого климатического режима может сигнализировать об аномальном периоде или проблеме данных.

## 2.8. Итоговая сводка и интерпретация

In [ ]:
summary_tables = {
    'classification_metrics': clf_results_df,
    'forecast_best_summary': best_forecast_summary,
    'residual_tests': residual_tests,
}

with pd.ExcelWriter(ARTIFACT_DIR / 'lab3_summary_tables.xlsx') as writer:
    for name, table in summary_tables.items():
        table.to_excel(writer, sheet_name=name[:31], index=False)

print('Итоговые артефакты сохранены в:', ARTIFACT_DIR.resolve())
print('\nСводка классификации:')
display(clf_results_df)
print('\nСводка прогноза:')
display(best_forecast_summary)
print('\nТесты остатков:')
display(residual_tests)

### Финальные выводы

1. Для метеоданных шести городов двухэтапный подход оправдан: сначала определяется климатический тип временного окна, затем применяется специализированная модель прогноза.
2. На этапе EDA выявлены сильная годовая сезонность, ненормальность распределений, различия температурных амплитуд и структуры осадков между городами.
3. Для обработки пропусков выбрана временная интерполяция на коротких промежутках, потому что метеопараметры меняются непрерывно. Выбросы не удалялись механически, так как экстремумы могут быть реальными опасными явлениями.
4. Для классификации лучше подходят модели на агрегированных оконных признаках, потому что они устойчивее сырых временных точек и лучше интерпретируются.
5. Для прогноза на 30 дней основную роль играют сезонные признаки, климатическая норма, текущая температура, лаги и скользящие статистики.
6. Качество прогноза зависит от города и горизонта: чем дальше горизонт, тем выше ошибка. Это соответствует физике атмосферы и ограниченной предсказуемости погоды.
7. Полученный notebook можно использовать как основу Git-репозитория: все этапы воспроизводимы, результаты сохраняются в `lab3_outputs`, зависимости перечислены в `requirements.txt`.